In [ ]:
#| default_exp ai

# AI

> Vertex AI generative models, Vector Search, and Vertex AI Search (RAG).

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part
    from google.cloud import aiplatform
    from google.cloud.aiplatform import MatchingEngineIndex, MatchingEngineIndexEndpoint
    from google.cloud import discoveryengine_v1 as discoveryengine
except ImportError:
    pass

In [ ]:
#| export
def _init_vertexai(auth):
    """Initialise Vertex AI SDK with auth credentials."""
    vertexai.init(project=auth.project, location=auth.region,
                  credentials=auth.credentials)

In [ ]:
#| export
def list_models(auth) -> list:
    """List available Vertex AI generative model publishers."""
    _init_vertexai(auth)
    # Returns well-known Gemini model IDs
    return [
        'gemini-1.5-pro',
        'gemini-1.5-flash',
        'gemini-2.0-flash-exp',
        'text-embedding-004',
    ]


def generate_content(
    auth,
    prompt: str,
    model: str = 'gemini-1.5-pro',
    max_tokens: int = 1024,
    **_,
) -> str:
    """Generate a text response from a Vertex AI Gemini model."""
    _init_vertexai(auth)
    m = GenerativeModel(model)
    response = m.generate_content(
        prompt,
        generation_config={'max_output_tokens': max_tokens},
    )
    return response.text

In [ ]:
#| export
def create_vector_search_index(
    auth,
    name: str,
    dimensions: int = 768,
    approximate_neighbors: int = 150,
    distance_measure: str = 'DOT_PRODUCT_DISTANCE',
    labels: dict = None,
    **_,
) -> dict:
    """Create a Vertex AI Vector Search (Matching Engine) index."""
    aiplatform.init(project=auth.project, location=auth.region,
                    credentials=auth.credentials)
    idx = MatchingEngineIndex.create_tree_ah_index(
        display_name=name,
        dimensions=dimensions,
        approximate_neighbors_count=approximate_neighbors,
        distance_measure_type=distance_measure,
        labels=labels or {},
    )
    return {'name': idx.resource_name, 'display_name': name}


def create_vector_search_endpoint(
    auth,
    name: str,
    public: bool = False,
    labels: dict = None,
    **_,
) -> dict:
    """Create a Vertex AI Vector Search index endpoint."""
    aiplatform.init(project=auth.project, location=auth.region,
                    credentials=auth.credentials)
    ep = MatchingEngineIndexEndpoint.create(
        display_name=name,
        public_endpoint_enabled=public,
        labels=labels or {},
    )
    return {'name': ep.resource_name, 'display_name': name}

In [ ]:
#| export
def create_search_app(
    auth,
    name: str,
    data_store_type: str = 'GENERIC',
    **_,
) -> dict:
    """Create a Vertex AI Search data store + app for RAG pipelines."""
    client = discoveryengine.DataStoreServiceClient(
        credentials=auth.credentials
    )
    parent = f'projects/{auth.project}/locations/global/collections/default_collection'
    data_store = discoveryengine.DataStore(
        display_name=name,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )
    op = client.create_data_store(
        parent=parent,
        data_store=data_store,
        data_store_id=name.replace(' ', '-').lower(),
    )
    result = op.result(timeout=300)
    return {'name': result.name, 'display_name': name}


def search_query(
    auth,
    app_id: str,
    query: str,
    page_size: int = 10,
) -> list:
    """Run a search query against a Vertex AI Search app."""
    client = discoveryengine.SearchServiceClient(
        credentials=auth.credentials
    )
    serving_config = (
        f'projects/{auth.project}/locations/global/collections/default_collection'
        f'/engines/{app_id}/servingConfigs/default_config'
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
    )
    response = client.search(request=req)
    return [r.document.struct_data for r in response.results]